# 02 - Tasks, depth oracles and the 2x2 splits
Builds the generators and depth oracles (`src/glwm2/data`), runs the flip checks, builds the
frozen splits with their SHA-256 hashes, and plots depth and length histograms per split.

In [ ]:
# parameters
N_CHECK = 300

In [ ]:
%matplotlib inline
import json, sys, time
from pathlib import Path
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from glwm2 import PROJECT_ROOT, RESULTS_DIR, PAPER_DIR
from glwm2.eval.plotting import set_style
set_style()
WRITTEN = []  # files this notebook writes

In [ ]:
from glwm2.data.addition import sample_addition
from glwm2.data.parity import sample_parity
from glwm2.data.depth import carry_depth, segment_depth, addition_rho
from glwm2.data.splits import build_frozen_splits, load_split, SEQ_SPLITS, MAZE_SPLITS, split_names
g = torch.Generator().manual_seed(0)
b = sample_addition(4, (6, 8), (3, 6), g)
for row, L, d in zip(b["x"].tolist(), b["length"].tolist(), b["depth"].tolist()):
    a = [t // 10 for t in row[:L]]; bb = [t % 10 for t in row[:L]]
    A = int("".join(map(str, a[::-1]))); B = int("".join(map(str, bb[::-1])))
    print(f"{A:>9d} + {B:>9d} = {A+B:>10d}   d* = {d}  (reference {carry_depth(a, bb)})")

In [ ]:
# flip checks (the full versions are tests/test_depth.py and tests/test_theory.py)
import subprocess
out = subprocess.run([sys.executable, "-m", "pytest", "-q", str(PROJECT_ROOT / "tests" / "test_depth.py"),
                      str(PROJECT_ROOT / "tests" / "test_data.py")], capture_output=True, text=True, cwd=PROJECT_ROOT)
print(out.stdout[-400:])
assert out.returncode == 0

In [ ]:
hashes = {t: build_frozen_splits(t) for t in ["addition", "parity", "mazes"]}
rows = []
for task in ["addition", "parity"]:
    for s in split_names(task):
        d = load_split(task, s)
        rows.append(dict(task=task, split=s, n=len(d["x"]), L_min=int(d["length"].min()), L_max=int(d["length"].max()),
                         d_min=int(d["depth"].min()), d_max=int(d["depth"].max()), sha256=hashes[task][s][:16]))
for s in split_names("mazes"):
    d = load_split("mazes", s)
    rows.append(dict(task="mazes", split=s, n=len(d["x"]), L_min=int(d["size"][0]), L_max=int(d["size"][0]),
                     d_min=int(d["depth"].min()), d_max=int(d["depth"].max()), sha256=hashes["mazes"][s][:16]))
splits_df = pd.DataFrame(rows); splits_df

In [ ]:
from glwm2.eval.plotting import savefig, SPLIT_COLORS
fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.6))
for ax, task in zip(axes, ["addition", "parity"]):
    for s in ["id_test", "ood_depth", "ood_length", "ood_both"]:
        d = load_split(task, s)
        ax.scatter(d["length"].numpy(), d["depth"].numpy(), s=3, alpha=0.4, color=SPLIT_COLORS[s], label=s)
    ax.set_xscale("log", base=2); ax.set_yscale("log", base=2)
    ax.set_xlabel("length L"); ax.set_ylabel("depth $d^*$"); ax.set_title(task)
axes[0].legend(markerscale=4, fontsize=7, frameon=False)
WRITTEN.append(savefig(fig, "fig2_splits"))

In [ ]:
fig, ax = plt.subplots(figsize=(3.4, 2.4))
for s in split_names("mazes"):
    d = load_split("mazes", s)
    ax.scatter(d["path_len"].numpy(), d["depth"].numpy(), s=3, alpha=0.4, label=s)
ax.set_xlabel("path length"); ax.set_ylabel("dead-end filling depth"); ax.legend(fontsize=6, markerscale=3, frameon=False)
WRITTEN.append(savefig(fig, "figA_maze_depths"))
from glwm2.utils.io import write_json
WRITTEN.append(write_json(RESULTS_DIR / "splits_summary.json", rows))
WRITTEN.append(PROJECT_ROOT / "data" / "frozen" / "HASHES.json")

In [ ]:
print("Files written by this notebook:")
for p in WRITTEN:
    print(" ", Path(p).relative_to(PROJECT_ROOT))